[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-08-pipe-mutate-resolve.ipynb#scrollTo=gg000001)

---
# Day 8 · Decorators Part 2: @pipe, @mutate, @resolve
**certified-journeys / hamilton-certified** · Day 8 · Advanced Decorators

> **Goal for today:** Use `@pipe` to chain sequential transforms on a single Series, `@mutate` to post-process an existing node's output, and `@resolve` to include or exclude nodes dynamically from the graph at build time.

In [ ]:
%pip install -q sf-hamilton

## Decorator Comparison

| Decorator | What it does | Best for |
|---|---|---|
| `@does(fn)` | Delegates body to `fn` | Same logic, different node names |
| `@pipe(step1, step2, ...)` | Chains transforms on one object | Sequential column cleaning |
| `@mutate(node_name=fn)` | Post-processes another node's output | Capping, clipping after the fact |
| `@resolve(...)` | Dynamically includes/excludes nodes | Feature flags, conditional features |

All four are in `hamilton.function_modifiers`. Today covers the last three.

In [ ]:
import sys, types
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag, pipe, mutate, resolve, step, value
from hamilton.plugins import h_pandas

rng = np.random.default_rng(8)
N = 200
raw_df = pd.DataFrame({
    'age':     np.concatenate([rng.integers(18, 75, N-10).astype(float), np.full(10, np.nan)]),
    'spend':   np.where(rng.random(N) < 0.08, -1.0, rng.exponential(100, N)),
    'score':   rng.uniform(0, 150, N),  # some out-of-range (>100)
})
print('Dataset shape:', raw_df.shape)
print('NaN ages:', raw_df['age'].isna().sum(),
      '| Negative spend:', (raw_df['spend'] < 0).sum(),
      '| score > 100:', (raw_df['score'] > 100).sum())

## Step 1 · @pipe — Chaining Sequential Transforms

`@pipe` applies a sequence of transformations to a single object. Each step receives the output of the previous one. The decorated function's parameter name is what gets piped through.

```python
@pipe(
    step(fn1),          # fn1(age) → Series
    step(fn2),          # fn2(result_of_fn1) → Series
    step(fn3, kwarg=5), # fn3(result_of_fn2, kwarg=5) → Series
)
def age_processed(age: pd.Series) -> pd.Series:
    ...
```

Use `@pipe` when transforms are **truly sequential on the same object** and you want them visible as a single named node in the graph. For independently testable steps, prefer separate functions.

In [ ]:
# Individual transform steps — each is a plain Python function
def _fill_median(s: pd.Series) -> pd.Series:
    return s.fillna(s.median())

def _clip_age(s: pd.Series, low: float = 18.0, high: float = 100.0) -> pd.Series:
    return s.clip(lower=low, upper=high)

def _zscore(s: pd.Series) -> pd.Series:
    std = s.std()
    return (s - s.mean()) / (std if std > 0 else 1.0)

def _clip_spend(s: pd.Series) -> pd.Series:
    return s.clip(lower=0)

def _log1p(s: pd.Series) -> pd.Series:
    return np.log1p(s)

# @pipe: chain fill → clip → zscore into one named node
@tag(feature_type='numerical')
@pipe(
    step(_fill_median),          # 1. fill NaN with median
    step(_clip_age, low=18.0, high=100.0),  # 2. clamp to [18, 100]
    step(_zscore),               # 3. z-score normalize
)
def age_processed(age: pd.Series) -> pd.Series:
    """Age: fill NaN → clip → z-score, in one pipeline node."""
    ...

# @pipe: chain clip → log for spend
@tag(feature_type='numerical')
@pipe(
    step(_clip_spend),  # 1. remove negatives
    step(_log1p),       # 2. log-transform
    step(_zscore),      # 3. z-score
)
def spend_processed(spend: pd.Series) -> pd.Series:
    """Spend: clip negatives → log1p → z-score."""
    ...

print('@pipe nodes defined: age_processed, spend_processed')

In [ ]:
pipe_module = types.ModuleType('pipe_demo')
pipe_module.age_processed  = age_processed
pipe_module.spend_processed = spend_processed
sys.modules['pipe_demo'] = pipe_module

dr_pipe = driver.Builder().with_modules(pipe_module).build()
result  = dr_pipe.execute(
    ['age_processed', 'spend_processed'],
    inputs={'age': raw_df['age'], 'spend': raw_df['spend']}
)

print('age_processed  — mean: {:+.4f}  std: {:.4f}  NaN: {}'.format(
    result['age_processed'].mean(), result['age_processed'].std(),
    result['age_processed'].isna().sum()))
print('spend_processed — mean: {:+.4f}  std: {:.4f}  NaN: {}'.format(
    result['spend_processed'].mean(), result['spend_processed'].std(),
    result['spend_processed'].isna().sum()))

### What just happened?
- **`@pipe` collapsed a 3-step process into one named DAG node** — the intermediate steps don't appear as separate nodes.
- **Mean ≈ 0, std ≈ 1** confirms the final z-score step ran correctly.
- **Zero NaNs** confirms the `_fill_median` step ran first, before the clip.
- The individual step functions (`_fill_median`, `_clip_age`, etc.) remain independently testable as plain Python.

## Step 2 · @mutate — Post-Processing Another Node

`@mutate` modifies the output of an **existing** node — without changing the original function. It's useful when you want to add a cap or clip as an afterthought, or when you own the downstream but not the upstream function.

```python
@mutate(existing_node=cap_function)
def existing_node(existing_node: pd.Series) -> pd.Series:
    ...
```

After `@mutate`, any node that depends on `existing_node` will see the mutated version. The original function's logic is unchanged.

In [ ]:
from hamilton.function_modifiers import mutate

def score_raw(score: pd.Series) -> pd.Series:
    """Raw score — may exceed 100 due to data quality."""
    return score

# Post-process: cap score_raw at 100 without touching the original function
def _cap_at_100(s: pd.Series) -> pd.Series:
    return s.clip(upper=100.0)

@mutate(score_raw=_cap_at_100)
def score_raw(score_raw: pd.Series) -> pd.Series:  # noqa: F811
    """score_raw after capping at 100."""
    ...

@tag(feature_type='numerical')
def score_normalized(score_raw: pd.Series) -> pd.Series:
    """Normalize the (already-capped) score to [0, 1]."""
    return score_raw / 100.0

mutate_module = types.ModuleType('mutate_demo')
mutate_module.score_raw        = score_raw
mutate_module.score_normalized = score_normalized
sys.modules['mutate_demo'] = mutate_module

dr_mut = driver.Builder().with_modules(mutate_module).build()
res    = dr_mut.execute(
    ['score_raw', 'score_normalized'],
    inputs={'score': raw_df['score']}
)

print('Original score max:      ', raw_df['score'].max().round(2))
print('score_raw max (mutated): ', res['score_raw'].max().round(2))
print('score_normalized range:  ', res['score_normalized'].min().round(3),
      '–', res['score_normalized'].max().round(3))
assert res['score_raw'].max() <= 100.0
print('✓ @mutate successfully capped score_raw at 100')

### What just happened?
- **`@mutate(score_raw=_cap_at_100)`** applied `_cap_at_100` to `score_raw`'s output — transparently to all downstream nodes.
- `score_normalized` receives the capped value without knowing a mutation happened.
- **Original `score_raw` function is unchanged** — `@mutate` is a non-invasive wrapper, useful when you can't modify the original module.

## Step 3 · @resolve — Dynamic Node Inclusion

`@resolve` determines at **graph build time** whether a node is included, based on a config value. Unlike `@config.when` (which picks between two implementations), `@resolve` can include or exclude a node entirely.

```python
@resolve(
    when=ResolveAt.CONFIG_EAGER,
    decorate_with=lambda enable_feature: tag(feature_type='boolean')
        if enable_feature else ...
)
```

The simpler pattern: `@resolve` with a function that returns a decorator or `None`.

In [ ]:
from hamilton.function_modifiers import resolve, ResolveAt

# Experimental feature: only include this node if 'enable_experimental=True' in config
@resolve(
    when=ResolveAt.CONFIG_EAGER,
    decorate_with=lambda enable_experimental: (
        tag(feature_type='boolean') if enable_experimental else tag(feature_type='boolean')
    )
)
def experimental_flag(age_processed: pd.Series, spend_processed: pd.Series) -> pd.Series:
    """Experimental: high-age + low-spend signal."""
    return ((age_processed > 1.0) & (spend_processed < -0.5)).astype(float)

# More practical @resolve pattern: use config to choose between two tag sets
# Here we show the graph-inclusion pattern with @config.when_not_in for completeness
from hamilton.function_modifiers import config as config_mod

@config_mod.when(enable_experimental=True)
def experimental_score__on(
    age_processed: pd.Series, spend_processed: pd.Series
) -> pd.Series:
    """Experimental feature — active when enable_experimental=True."""
    return (age_processed * -1) + spend_processed  # high age, low spend → high risk

@config_mod.when(enable_experimental=False)
def experimental_score__off(
    spend_processed: pd.Series
) -> pd.Series:
    """Fallback when experimental feature is disabled."""
    return spend_processed * 0  # return zeros — neutral signal

resolve_module = types.ModuleType('resolve_demo')
resolve_module.age_processed        = age_processed
resolve_module.spend_processed      = spend_processed
resolve_module.experimental_score__on  = experimental_score__on
resolve_module.experimental_score__off = experimental_score__off
sys.modules['resolve_demo'] = resolve_module

# Build with experimental ON
dr_on = (
    driver.Builder()
    .with_modules(resolve_module)
    .with_config({'enable_experimental': True})
    .build()
)

# Build with experimental OFF
dr_off = (
    driver.Builder()
    .with_modules(resolve_module)
    .with_config({'enable_experimental': False})
    .build()
)

r_on  = dr_on.execute(['experimental_score'], inputs={'age': raw_df['age'], 'spend': raw_df['spend']})
r_off = dr_off.execute(['experimental_score'], inputs={'age': raw_df['age'], 'spend': raw_df['spend']})

print('experimental ON  — non-zero values:', (r_on['experimental_score'] != 0).sum())
print('experimental OFF — all zeros?       ', (r_off['experimental_score'] == 0).all())

### What just happened?
- **`@config.when`** here acts as a feature flag: enable or disable a whole node by changing one config value.
- The disabled branch returns zeros — a neutral signal that doesn't affect any downstream model.
- This pattern is production-safe: **roll back an experimental feature by changing config**, not by deploying new code.

## Step 4 · When to Use Each Decorator

Decorator choice depends on the situation — picking the wrong one creates unnecessary complexity.

In [ ]:
# Decision guide — run this as a reference
scenarios = [
    ('Same logic, different inputs (age_norm, spend_norm, tenure_norm)',
     '@does(_zscore)'),
    ('Multi-step column cleaning (fill → clip → scale)',
     '@pipe(step(fill), step(clip), step(scale))'),
    ('Add a cap/clip to an existing node without editing it',
     '@mutate(existing_node=_cap_fn)'),
    ('Include a node only when a feature flag is enabled',
     '@config.when(feature_flag=True)'),
    ('Dynamically choose decorator based on config at build time',
     '@resolve(when=CONFIG_EAGER, decorate_with=lambda cfg: ...)'),
    ('Two completely different implementations of the same output',
     '@config.when for each variant'),
]

print('DECORATOR DECISION GUIDE')
print('=' * 60)
for scenario, answer in scenarios:
    print(f'Scenario: {scenario}')
    print(f'Use:      {answer}')
    print()

### What just happened?
- The decision guide is the key takeaway from Days 3 and 8 combined.
- **`@pipe` vs separate functions**: use `@pipe` when the steps are not independently meaningful (intermediate results would just be noise in the graph); use separate functions when each step is a named concept your team reasons about.
- **`@mutate` vs modifying the original function**: prefer `@mutate` when you don't own the original module, or when the post-processing is a cross-cutting concern (like capping all scores).

In [ ]:
# Challenge: rewrite the Day 6 normalization approach using @pipe instead of @does
# Build age_pipe, spend_pipe, tenure_pipe each using:
#   @pipe(step(_fill_median), step(_zscore))
# Then compare: which approach is clearer for a team maintaining the code?

# @tag(feature_type='numerical')
# @pipe(step(_fill_median), step(_zscore))
# def age_pipe(age: pd.Series) -> pd.Series: ...

print('Implement the @pipe variants and compare with the @does approach from Day 6!')

---
## Day 8 key concepts recap

| Decorator | Key rule |
|---|---|
| `@pipe(step(fn), ...)` | Sequential steps on one object; intermediate results are invisible in the graph |
| `@mutate(node=fn)` | Non-invasive post-processing; downstream sees mutated value automatically |
| `@config.when` as feature flag | Cleanest pattern for conditional node inclusion |
| `@resolve` | Use when the decorator itself depends on config, not just the implementation |
| Step functions (`_underscore`) | Not registered as nodes; independently testable helpers for `@pipe` |

> **Tip:** `@pipe` is a power tool — it makes long transformation chains readable. Use it only when the steps are truly sequential on the same object; otherwise separate functions are clearer.

---
## What's next
**Day 9** → Async and parallel execution: rewrite a synchronous pipeline as async, fan out with the parallel executor, and benchmark the speedup.

Mark Day 8 complete in your [tracker](../index.html).